In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

vh_df = pd.read_csv('/Users/kimlanaghen/Downloads/Vaccine_Hesitancy_for_COVID-19__County_and_local_estimates.csv')

In [3]:
print(vh_df.head())
print(vh_df.info()) 
print(vh_df.columns)

   FIPS Code                     County Name    State  Estimated hesitant  \
0       1131          Wilcox County, Alabama  ALABAMA                0.23   
1       1129      Washington County, Alabama  ALABAMA                0.23   
2       1133         Winston County, Alabama  ALABAMA                0.22   
3       1127          Walker County, Alabama  ALABAMA                0.23   
4       2013  Aleutians East Borough, Alaska   ALASKA                0.26   

   Estimated strongly hesitant  Social Vulnerability Index (SVI)  \
0                         0.11                              0.93   
1                         0.11                              0.73   
2                         0.11                              0.70   
3                         0.11                              0.75   
4                         0.12                              0.58   

        SVI Category  CVAC level of concern for vaccination rollout  \
0  Very High Concern                                     

Checking for missing values

In [4]:
#checking for missing values in each column
print("Percent missing by column:")
print(vh_df.isna().mean() * 100)

Percent missing by column:
FIPS Code                                                 0.000000
County Name                                               0.000000
State                                                     0.000000
Estimated hesitant                                        0.000000
Estimated strongly hesitant                               0.000000
Social Vulnerability Index (SVI)                          0.031827
SVI Category                                              0.000000
CVAC level of concern for vaccination rollout             0.000000
CVAC Level Of Concern                                     0.000000
Percent adults fully vaccinated against COVID-19         10.057288
Percent Hispanic                                          0.000000
Percent non-Hispanic American Indian/Alaska Native        0.000000
Percent non-Hispanic Asian                                0.000000
Percent non-Hispanic Black                                0.000000
Percent non-Hispanic Native Hawaiia

In [5]:
from IPython.display import FileLink

# take first 50 rows and save as CSV for download
subset = vh_df.iloc[:50]
outfile = 'vh_first50.csv'
subset.to_csv(outfile, index=False)

FileLink(outfile)

/Users/kimlanaghen/Downloads/vh_first50.csv

Conformance (is all data the same notation?)

In [7]:
;# check if FIPS codes are all 5 digits
fips_cols = [c for c in vh_df.columns if 'fips' in c.lower()]
if not fips_cols:
    print("No FIPS-like column found.")
else:
    fips_col = fips_cols[0]
    s = vh_df[fips_col]
    num = pd.to_numeric(s, errors='coerce')
    s_norm = s.astype(str).str.strip().copy()
    mask_num = num.notna()
    s_norm.loc[mask_num] = num[mask_num].astype(int).astype(str).str.zfill(5)
    missing_mask = s.isna() | s.astype(str).str.strip().str.lower().isin(['nan','none',''])
    valid_mask = s_norm.str.match(r'^\d{5}$') & ~missing_mask

    print("FIPS column:", fips_col)
    print("Total rows:", len(s))
    print("Missing/blank:", int(missing_mask.sum()))
    print("Valid 5-digit FIPS:", int(valid_mask.sum()))
    print("Invalid (not 5 digits):", int((~valid_mask & ~missing_mask).sum()))

    if (~valid_mask & ~missing_mask).any():
        print("\nExamples of invalid FIPS values:")
        print(vh_df.loc[~valid_mask & ~missing_mask, fips_col].drop_duplicates().head(20).to_list())
        print("\nRows with invalid FIPS (up to 20):")
        print(vh_df.loc[~valid_mask & ~missing_mask].head(20))

FIPS column: FIPS Code
Total rows: 3142
Missing/blank: 0
Valid 5-digit FIPS: 3142
Invalid (not 5 digits): 0


***Timeliness in this dataset is not applicable because there are no timestamps***

Duplication Check in the FIPS Code and County Names

In [6]:
# detect FIPS and county columns (case-insensitive) and report duplicates
fips_cols = [c for c in vh_df.columns if 'fips' in c.lower()]
county_cols = [c for c in vh_df.columns if 'county' in c.lower()]

if not fips_cols or not county_cols:
    print("Could not find required columns.")
    print("FIPS-like columns found:", fips_cols)
    print("County-like columns found:", county_cols)
else:
    fips_col = fips_cols[0]
    county_col = county_cols[0]
    print("Using columns -> FIPS:", fips_col, ", County:", county_col)

    # duplicates by FIPS value
    dup_fips_counts = vh_df[fips_col].value_counts(dropna=True)
    dup_fips_more_than_one = dup_fips_counts[dup_fips_counts > 1]
    print("\nFIPS values appearing >1:", len(dup_fips_more_than_one))
    if not dup_fips_more_than_one.empty:
        print(dup_fips_more_than_one.head(10))
        print("Example rows with duplicated FIPS (up to 20):")
        print(vh_df[vh_df[fips_col].isin(dup_fips_more_than_one.index)].sort_values([fips_col, county_col]).head(20))

    # duplicates by county name
    dup_county_counts = vh_df[county_col].value_counts(dropna=True)
    dup_county_more_than_one = dup_county_counts[dup_county_counts > 1]
    print("\nCounty names appearing >1:", len(dup_county_more_than_one))
    if not dup_county_more_than_one.empty:
        print(dup_county_more_than_one.head(10))
        print("Example rows with duplicated county names (up to 20):")
        print(vh_df[vh_df[county_col].isin(dup_county_more_than_one.index)].sort_values([county_col, fips_col]).head(20))

    # duplicated (FIPS, county) pairs
    dup_pair_mask = vh_df.duplicated(subset=[fips_col, county_col], keep=False)
    print("\nRows with duplicated (FIPS, County) pair:", dup_pair_mask.sum())
    if dup_pair_mask.any():
        print(vh_df[dup_pair_mask].sort_values([fips_col, county_col]).head(20))

Using columns -> FIPS: FIPS Code , County: County Name

FIPS values appearing >1: 0

County names appearing >1: 0

Rows with duplicated (FIPS, County) pair: 0
